In [6]:
import os
import random
import gc
import signal
import shutil
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
import imageio.v2 as imageio
import networkx as nx
import seaborn as sns
from PIL import Image as _PILImage
from collections import Counter
from dataclasses import dataclass
from math import exp
from typing import TypeVar, Callable

from torch_geometric.datasets import QM9
from torch_geometric.transforms import ToDense
from torch.utils.data import DataLoader as TorchDataLoader

if not hasattr(torch.utils.data.dataloader, 'T_co'):
    torch.utils.data.dataloader.T_co = TypeVar('T_co', covariant=True)

from schnetpack.nn import Dense, scatter_add
from schnetpack.nn.activations import shifted_softplus
import schnetpack.nn as spk_nn

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

In [8]:
# Configuration and Hyperparameters
SEED = 100

DATA_ROOT = os.environ.get("QM9_ROOT", "/home/chaitanya/TPNN/QM9/dataset")
SAVE_DIR = 'mincut_viz'

# Set seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(SAVE_DIR, exist_ok=True)

In [9]:
BATCH_SIZE = 32
MIN_NODES = 26  # Filter for graphs with > 25 nodes

# Load raw dataset
dataset = QM9(root=DATA_ROOT)

# Filter for graphs with > MIN_NODES nodes
filtered_data = [d for d in dataset if d.num_nodes > MIN_NODES]
print(f"Total QM9 molecules: {len(dataset)}")
print(f"Filtered dataset size (> {MIN_NODES} nodes): {len(filtered_data)}")

# Find the maximum number of nodes across the filtered set
MAX_NUM_NODES = max(d.num_nodes for d in filtered_data)
print(f"Maximum number of nodes in filtered dataset: {MAX_NUM_NODES}")

# Convert every graph to dense format, padded to MAX_NUM_NODES
to_dense = ToDense(num_nodes=MAX_NUM_NODES)
dense_data = []
for d in filtered_data:
    d = to_dense(d)
    # Fix 1: ToDense doesn't pad `z` — pad it manually to [MAX_NUM_NODES]
    orig_n = d.z.shape[0]
    if orig_n < MAX_NUM_NODES:
        d.z = torch.cat([d.z, torch.zeros(MAX_NUM_NODES - orig_n, dtype=d.z.dtype)])
    # Fix 2: QM9 has 4D edge_attr → adj is [N, N, 4]. Collapse to binary [N, N]
    if d.adj.dim() == 3:
        d.adj = (d.adj.sum(-1) > 0).float()
    # Precompute pairwise bond lengths (Euclidean distances) for connected atoms
    pairwise_dists = torch.cdist(d.pos.unsqueeze(0), d.pos.unsqueeze(0)).squeeze(0)
    d.edge_dists = pairwise_dists * d.adj  # keep only distances where bonds exist
    dense_data.append(d)

# Verify a single sample
d0 = dense_data[0]
print(f"Sample shapes: z={d0.z.shape}, pos={d0.pos.shape}, adj={d0.adj.shape}, mask={d0.mask.shape}, edge_dists={d0.edge_dists.shape}")

# --- Dense-aware collation ---
class DenseBatch:
    """Lightweight batch container with .to(device) support."""
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
    def to(self, device):
        for k in list(self.__dict__):
            v = getattr(self, k)
            if isinstance(v, torch.Tensor):
                setattr(self, k, v.to(device))
        return self

def dense_collate(data_list):
    return DenseBatch(
        z    = torch.stack([d.z    for d in data_list]),
        pos  = torch.stack([d.pos  for d in data_list]),
        adj  = torch.stack([d.adj  for d in data_list]),
        mask = torch.stack([d.mask for d in data_list]),
        edge_dists = torch.stack([d.edge_dists for d in data_list]),
    )

# Split Data
n = len(dense_data)
train_size = int(0.8 * n)
val_size = int(0.1 * n)
test_size = n - train_size - val_size

train_dataset = dense_data[:train_size]
val_dataset   = dense_data[train_size:train_size + val_size]
test_dataset  = dense_data[train_size + val_size:]

# Create Loaders
train_loader = TorchDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=dense_collate)
val_loader   = TorchDataLoader(val_dataset,   batch_size=1,          shuffle=False, collate_fn=dense_collate)
test_loader  = TorchDataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=dense_collate)

print(f"Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"Each sample is padded to {MAX_NUM_NODES} nodes (dense format).")

Total QM9 molecules: 130831
Filtered dataset size (> 26 nodes): 391
Maximum number of nodes in filtered dataset: 29
Sample shapes: z=torch.Size([29]), pos=torch.Size([29, 3]), adj=torch.Size([29, 29]), mask=torch.Size([29]), edge_dists=torch.Size([29, 29])
Train/Val/Test sizes: 312/39/40
Each sample is padded to 29 nodes (dense format).


In [7]:
class SchNetInteraction(nn.Module):
    r"""SchNet interaction block for modeling interactions of atomistic systems."""

    def __init__(
        self,
        n_atom_basis: int,
        n_rbf: int,
        n_filters: int,
        activation: Callable = shifted_softplus,
    ):
        super(SchNetInteraction, self).__init__()
        self.in2f = Dense(n_atom_basis, n_filters, bias=False, activation=None)
        self.f2out = nn.Sequential(
            Dense(n_filters, n_atom_basis, activation=activation),
            Dense(n_atom_basis, n_atom_basis, activation=None),
        )
        self.filter_network = nn.Sequential(
            Dense(n_rbf, n_filters, activation=activation), Dense(n_filters, n_filters)
        )

    def reset_parameters(self):
        for module in self.modules():
            if module is not self and hasattr(module, 'reset_parameters'):
                module.reset_parameters()

    def forward(self, x, f_ij, idx_i, idx_j, rcut_ij):
        x = self.in2f(x)
        Wij = self.filter_network(f_ij)
        Wij = Wij * rcut_ij[:, None]
        x_j = x[idx_j]
        x_ij = x_j * Wij
        x = scatter_add(x_ij, idx_i, dim_size=x.shape[0])
        x = self.f2out(x)
        return x

In [36]:
class SchNetNodeFeatures(torch.nn.Module):
    def __init__(self, hidden_channels=128, n_filters=64, n_interactions=1, n_rbf=50, schnet_cutoff=10.0):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.cutoff = schnet_cutoff
        self.n_rbf = n_rbf
        self.embedding = torch.nn.Embedding(100, hidden_channels)
        self.radial_basis = spk_nn.GaussianRBF(n_rbf=n_rbf, cutoff=schnet_cutoff)
        self.cutoff_fn = spk_nn.CosineCutoff(schnet_cutoff)
        self.interactions = torch.nn.ModuleList([
            SchNetInteraction(n_atom_basis=hidden_channels, n_rbf=n_rbf, n_filters=n_filters)
            for _ in range(n_interactions)
        ])

    def reset_parameters(self):
        self.embedding.reset_parameters()
        for interaction in self.interactions:
            interaction.reset_parameters()

    def forward(self, z, pos, mask=None, atom_embedding=None):
        B, N = z.shape
        if mask is None:
            mask = torch.ones_like(z, dtype=torch.bool)
        if atom_embedding is not None:
            h = atom_embedding
        else:
            h = self.embedding(z)
        valid_mask = mask.view(-1)
        h_flat = h.reshape(-1, self.hidden_channels)[valid_mask]
        dist = torch.cdist(pos, pos)
        mask_i = mask.unsqueeze(2)
        mask_j = mask.unsqueeze(1)
        diag_mask = torch.eye(N, device=pos.device).unsqueeze(0)
        adj = (dist <= self.cutoff) & (mask_i * mask_j).bool() & (diag_mask == 0)
        edge_indices = adj.nonzero()
        b_idx, i_local, j_local = edge_indices[:, 0], edge_indices[:, 1], edge_indices[:, 2]
        flat_indices = torch.full((B, N), -1, dtype=torch.long, device=pos.device)
        flat_indices[mask] = torch.arange(valid_mask.sum(), device=pos.device)
        idx_i = flat_indices[b_idx, i_local]
        idx_j = flat_indices[b_idx, j_local]
        Rij = pos[b_idx, j_local] - pos[b_idx, i_local]
        d_ij = torch.norm(Rij, dim=-1)
        f_ij = self.radial_basis(d_ij)
        rcut_ij = self.cutoff_fn(d_ij)
        for interaction in self.interactions:
            v = interaction(h_flat, f_ij, idx_i, idx_j, rcut_ij)
            h_flat = h_flat + v
        out = torch.zeros((B, N, self.hidden_channels), device=z.device, dtype=h.dtype)
        out[mask] = h_flat
        return out


class TypeMapping(nn.Module):
    def __init__(self, n_atom_basis, n_clusters):
        super().__init__()
        self.n_clusters = n_clusters
        self.lin1 = nn.Linear(n_atom_basis, n_clusters)
        self.act = nn.SiLU()
        self.lin2 = nn.Linear(n_clusters, n_clusters)
        self.softmax = nn.Softmax(dim=-1)
        nn.init.xavier_uniform_(self.lin1.weight)
        nn.init.zeros_(self.lin1.bias)
        nn.init.orthogonal_(self.lin2.weight)
        nn.init.zeros_(self.lin2.bias)

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin1.weight)
        nn.init.zeros_(self.lin1.bias)
        nn.init.orthogonal_(self.lin2.weight)
        nn.init.zeros_(self.lin2.bias)

    def forward(self, x, mask=None):
        s = self.softmax(self.lin2(self.act(self.lin1(x))))
        if mask is not None:
            s = s * mask.unsqueeze(-1)
        return s


class SoftBFS(nn.Module):
    def __init__(self, max_iters=5, temperature=10.0):
        super(SoftBFS, self).__init__()
        self.max_iters = max_iters
        self.temperature = temperature

    def forward(self, S_type, A_cov):
        S_sim = torch.matmul(S_type, S_type.transpose(-2, -1))
        C_0 = S_sim * A_cov
    
        C_prime = C_0.clone()
        for _ in range(self.max_iters):

            C_prime = torch.matmul(C_0, C_prime)
            C_prime = torch.sigmoid(self.temperature * (C_prime - 0.5))
        return C_prime


class DynamicCoarseGraining(nn.Module):
    """Differentiable coarse-graining layer using soft BFS similarity."""

    def __init__(self, max_clusters, threshold=0.5):
        super().__init__()
        self.max_clusters = max_clusters
        self.threshold = threshold

    def _find_cluster_representatives(self, C_soft, node_mask):
        B, N, _ = C_soft.shape
        L = self.max_clusters
        device = C_soft.device
        rep_indices = torch.zeros(B, L, dtype=torch.long, device=device)
        cluster_mask = torch.zeros(B, L, dtype=torch.bool, device=device)
        C_hard = (C_soft > self.threshold).float()
        hash_vec = torch.randn(N, device=device)
        row_hashes = C_hard @ hash_vec
        for b in range(B):
            valid = node_mask[b]
            hashes_b = row_hashes[b][valid]
            if hashes_b.numel() == 0:
                continue
            unique_h, inverse = torch.unique(hashes_b, return_inverse=True)
            n_clusters = min(L, len(unique_h))
            reps = torch.zeros(len(unique_h), dtype=torch.long, device=device)
            reps.scatter_(0, inverse, torch.arange(len(hashes_b), device=device))
            valid_indices = valid.nonzero(as_tuple=False).squeeze(-1)
            rep_indices[b, :n_clusters] = valid_indices[reps[:n_clusters]]
            cluster_mask[b, :n_clusters] = True
        return rep_indices, cluster_mask

    def forward(self, C_soft, X_atom, A_atom, node_mask):
        B, N, _ = C_soft.shape
        L = self.max_clusters
        pair_mask = node_mask.float().unsqueeze(-1) * node_mask.float().unsqueeze(-2)
        C_soft = C_soft * pair_mask
        with torch.no_grad():
            rep_indices, cluster_mask = self._find_cluster_representatives(C_soft, node_mask)
        idx_exp = rep_indices.unsqueeze(-1).expand(B, L, N)
        S_T = torch.gather(C_soft, 1, idx_exp)
        S_spatial = S_T.transpose(1, 2)
        active = node_mask.unsqueeze(-1) & cluster_mask.unsqueeze(1)
        S_spatial = S_spatial * active.float()
        S_T = S_spatial.transpose(1, 2)
        S_T_norm = S_T / S_T.sum(dim=-1, keepdim=True).clamp(min=1e-8)
        S_T_norm = S_T_norm * active.transpose(1, 2).float()
        X_cg = torch.bmm(S_T_norm, X_atom)
        A_cg = torch.bmm(torch.bmm(S_T_norm, A_atom), S_spatial)
        ind = torch.arange(L, device=A_cg.device)
        A_cg[:, ind, ind] = 0
        # deg = A_cg.sum(dim=-1).clamp(min=1e-15)
        # d_inv_sqrt = deg.sqrt().unsqueeze(1)
        # A_cg = A_cg / d_inv_sqrt / d_inv_sqrt.transpose(1, 2)
        return X_cg, A_cg, S_spatial, cluster_mask


# ══════════════════════════════════════════════════════════════
# MincutTopoLayer  – single coarse-graining pass
# ══════════════════════════════════════════════════════════════
class MincutTopoLayer(nn.Module):
    """Single hierarchical coarse-graining layer.

    Takes (z, pos, adj, mask) and produces coarse-grained outputs
    (cg_z, cg_pos, cg_adj, cg_mask) that can feed into the next layer.
    """
    def __init__(self, hidden_channels=128, n_filters=64, n_interactions=3,
                 n_clusters=100, schnet_cutoff=10.0, r_min_cutoff=2.0,
                 r_dist_cutoff=3.5, max_beads=16, max_iters=5, temperature=10.0, use_embedding=True):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.use_embedding = use_embedding
        self.max_iters = max_iters
        self.temperature = temperature
        # Encoder (interaction blocks; embedding only used in first layer)
        self.encoder = SchNetNodeFeatures(
            hidden_channels=hidden_channels,
            n_filters=n_filters,
            n_interactions=n_interactions,
            schnet_cutoff=schnet_cutoff
        )

        # Type Mapping
        self.type_mapping = TypeMapping(hidden_channels, n_clusters)

        # Cutoff functions
        self.mincut_cutoff_fn = spk_nn.CosineCutoff(r_min_cutoff)
        self.proximity_fn = spk_nn.CosineCutoff(r_dist_cutoff)

        # SoftBFS and Dynamic Coarse-Graining
        self.softBFS = SoftBFS(max_iters=self.max_iters, temperature=self.temperature)
        self.coarse_grain_layer = DynamicCoarseGraining(max_clusters=max_beads, threshold=0.5)
        self.h_norm = nn.LayerNorm(hidden_channels)
        
    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.type_mapping.reset_parameters()
        self.h_norm.reset_parameters()
        
    def forward(self, z, pos, adj, mask=None, node_features=None):
        B, N = z.shape[:2]
        if mask is None:
            mask = torch.ones(B, N, dtype=torch.bool, device=z.device)

        # 1. Encode
        # Layer 0: Embedding → Interaction blocks
        # Layer 1+: node_features (cg_x from prev layer) → Interaction blocks
        if node_features is not None:
            h = self.encoder(z, pos, mask, atom_embedding=node_features)
        else:
            h = self.encoder(z, pos, mask)
        h = self.h_norm(h)
        # 2. Type Assignment
        s_type = self.type_mapping(h, mask)
        s_type_hard = torch.argmax(s_type, dim=-1)
        K = s_type.shape[-1]
        s_type_h = torch.zeros_like(s_type)
        s_type_h.scatter_(2, s_type_hard.unsqueeze(-1), 1.0)
        s_type_h = s_type_h * mask.unsqueeze(-1)

        # 3. Type Similarity
        type_sim = torch.matmul(s_type, s_type.transpose(1, 2))
        type_sim_h = torch.matmul(s_type_h, s_type_h.transpose(1, 2))

        # 4. Distance-dependent adjacencies
        dist = torch.cdist(pos, pos)
        proximity_adj = self.proximity_fn(dist)
        a_cov = adj + torch.eye(N, device=pos.device).unsqueeze(0)
        mincut_adj = self.mincut_cutoff_fn(dist)

        atom_mask_2d = mask.float().unsqueeze(2) * mask.float().unsqueeze(1)
        proximity_adj = proximity_adj * atom_mask_2d
        mincut_adj = mincut_adj * atom_mask_2d
        a_cov = a_cov * atom_mask_2d
        
        a_cov_bin = (a_cov > 1e-5).float() * atom_mask_2d # ensure binary and masked (for subsequent layers) 
        # 5. Soft BFS
        c_prime = self.softBFS(s_type, a_cov_bin)
        if mask is not None:
            c_prime = c_prime * atom_mask_2d

        # 6. Moiety similarities
        c_matrix_soft = type_sim * proximity_adj
        c_h = type_sim_h * a_cov_bin

        # 7. Dynamic Coarse-Graining
        out_x, out_adj, s_space, cluster_mask = self.coarse_grain_layer(c_prime, h, adj, mask)

        # 8. Hard spatial assignments
        s_space_hard = torch.argmax(s_space, dim=-1)
        M = s_space.shape[-1]
        s_space_h = torch.zeros_like(s_space)
        s_space_h.scatter_(2, s_space_hard.unsqueeze(-1), 1.0)
        s_space_h = s_space_h * mask.unsqueeze(-1) * cluster_mask.unsqueeze(1).float()

        # 9. Hard coarsened adjacency
        out_adj_h = torch.matmul(torch.matmul(s_space_h.transpose(1, 2), adj), s_space_h)
        ind_h = torch.arange(M, device=out_adj_h.device)
        out_adj_h[:, ind_h, ind_h] = 0
        # If two moieties share at least 1 covalent bond, they are connected (1.0).
        out_adj_h = (out_adj_h > 0.5).float()

        # 10. Center of Mass (mass ≈ 2*Z)
        mass = 2 * z.unsqueeze(-1).float()
        weighted_pos = pos * mass
        numerator = torch.matmul(s_space_h.transpose(1, 2), weighted_pos)
        denominator = torch.matmul(s_space_h.transpose(1, 2), mass)
        cluster_com = numerator / (denominator + 1e-6)

        # 11. Coarse-grained z (sum of atomic numbers per cluster)
        cg_z = torch.matmul(s_space_h.transpose(1, 2), z.unsqueeze(-1).float()).squeeze(-1)

        return {
            "original_node_features": h,
            "type_assignments": s_type,
            "type_assignments_hard": s_type_h,
            "spatial_assignments": s_space,
            "spatial_assignments_hard": s_space_h,
            "adjacency_dist": proximity_adj,
            "adjacency_min": mincut_adj,
            "adjacency_mol": adj,
            "coarse_node_features": out_x,
            "coarse_adjacency": out_adj,
            "coarse_adjacency_hard": out_adj_h,
            "cluster_com": cluster_com,
            "cluster_mask": cluster_mask,
            "type_similarity": type_sim,
            "atom_mask": mask,
            "distance_dependent_type_similarity_soft": c_matrix_soft,
            "distance_dependent_type_similarity_hard": c_h,
            "softbfs": c_prime,
            "cg_z": cg_z,
            "cg_pos": cluster_com,
            "cg_adj": out_adj_h,
            "cg_mask": cluster_mask,
        }
# ══════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════
# HierarchicalMincutTopoModel – n stacked coarse-graining layers
# ══════════════════════════════════════════════════════════════
class HierarchicalMincutTopoModel(nn.Module):
    """Multi-layer hierarchical coarse-graining model.

    Applies MincutTopoLayer n times, feeding the coarse-grained output
    of each layer into the next.

    Args:
        n_layers:  Number of hierarchical coarse-graining layers (default 2).
        layer_configs:  Optional list of dicts with per-layer overrides.
            Supported keys: hidden_channels, n_filters, n_interactions,
            n_clusters, schnet_cutoff, r_min_cutoff, r_dist_cutoff, max_beads.
    """
    def __init__(self, n_layers=2, hidden_channels=128, n_filters=64, n_interactions=2,
                 n_clusters=100, schnet_cutoff=10.0, r_min_cutoff=2.0,
                 r_dist_cutoff=3.5, max_beads=16, max_iters=5, temperature=10.0, layer_configs=None):
        super().__init__()

        default_cfg = dict(
            hidden_channels=hidden_channels, n_filters=n_filters,
            n_interactions=n_interactions, n_clusters=n_clusters,
            schnet_cutoff=schnet_cutoff, r_min_cutoff=r_min_cutoff,
            r_dist_cutoff=r_dist_cutoff, max_beads=max_beads,
            max_iters=max_iters, temperature=temperature
        )

        self.layers = nn.ModuleList()
        for i in range(n_layers):
            cfg = {**default_cfg}
            if layer_configs and i < len(layer_configs):
                cfg.update(layer_configs[i])
            layer = MincutTopoLayer(use_embedding=(i == 0), **cfg)
            self.layers.append(layer)

    def reset_parameters(self):
        for layer in self.layers:
            layer.reset_parameters()

    def forward(self, z, pos, adj, mask=None):
        layer_results = []
        node_features = None  # first layer uses embedding
        for layer in self.layers:
            result = layer(z, pos, adj, mask=mask, node_features=node_features)
            layer_results.append(result)
            # Feed CG outputs to next layer
            z = result["cg_z"]
            pos = result["cg_pos"]
            adj = result["cg_adj"]
            mask = result["cg_mask"]
            node_features = result["coarse_node_features"]  # cg_x → interaction blocks of next layer
        return layer_results

    def get_hard_moieties(self, z, pos, adj, mask=None):
        with torch.no_grad():
            layer_results = self(z, pos, adj, mask=mask)
            all_moieties = []
            for result in layer_results:
                c_h = result["distance_dependent_type_similarity_hard"]
                layer_mask = result["atom_mask"]
                B, N, _ = c_h.shape
                batch_moieties = []
                c_h_cpu = c_h.cpu().numpy()
                mask_cpu = layer_mask.cpu().numpy()
                for b in range(B):
                    num_atoms = int(mask_cpu[b].sum())
                    a = c_h_cpu[b, :num_atoms, :num_atoms]
                    _, labels = connected_components(csr_matrix(a), directed=False, return_labels=True)
                    batch_moieties.append(labels)
                all_moieties.append(batch_moieties)
            return layer_results, all_moieties

In [11]:
# ══════════════════════════════════════════════════════════════
# Loss Configuration – per-layer coefficients & warmups
# ══════════════════════════════════════════════════════════════

@dataclass
class LayerLossConfig:
    """Per-layer loss hyperparameters.

    Weights
    -------
    w_cut_moinn   : MoINN cut loss weight
    w_ortho_moinn : MoINN orthogonality loss weight
    w_ent_moinn   : MoINN entropy loss weight
    w_recon_pool  : Pooling reconstruction loss weight
    w_cut_pool    : Pooling MinCut loss weight
    w_ortho_pool  : Pooling orthogonality loss weight
    w_link_pool   : Pooling link loss weight
    w_ent_pool    : Pooling entropy loss weight

    Warmup Epochs (sigmoid ramp from 0 → 1)
    ----------------------------------------
    warmup_*      : Epochs to ramp the corresponding loss

    Scheduling
    ----------
    pool_start_epoch : Epoch at which pooling losses activate
    """
    # MoINN loss weights
    w_cut_moinn:   float = 1.0
    w_ortho_moinn: float = 1.0
    w_ent_moinn:   float = 0.20

    # Pooling loss weights
    w_recon_pool:  float = 0.0
    w_cut_pool:    float = 0.0
    w_ortho_pool:  float = 0.0
    w_link_pool:   float = 0.0
    w_ent_pool:    float = 0.0

    # Warmup durations (epochs)
    warmup_cut_moinn:  int = 10
    warmup_ent_moinn:  int = 20
    warmup_ortho_moinn:int = 0
    
    warmup_recon_pool: int = 10
    warmup_cut_pool:   int = 0
    warmup_ortho_pool: int = 0
    warmup_link_pool:  int = 10
    warmup_ent_pool:   int = 20

    # Scheduling
    pool_start_epoch: int = 50


class HierarchicalLossConfig:
    """Multi-layer loss configuration with individually settable per-layer configs."""
    def __init__(self, n_layers=2, layer_configs=None):
        if layer_configs is not None:
            self.layer_configs = list(layer_configs)
        else:
            self.layer_configs = [LayerLossConfig() for _ in range(n_layers)]


# ══════════════════════════════════════════════════════════════
# Batch Tensor Utilities
# ══════════════════════════════════════════════════════════════
def batch_operation(op, a, b=None):
    if b is None:
        return torch.stack([op(a[i]) for i in range(a.shape[0])])
    return torch.stack([op(a[i], b[i]) for i in range(a.shape[0])])

def batch_trace(x):   return batch_operation(torch.trace, x)
def batch_norm(x):    return batch_operation(torch.norm, x)
def batch_diag(x):    return batch_operation(torch.diag, x)
def batch_div(a, b):  return batch_operation(torch.div, a, b)


def normalize(adj, node_mask=None, threshold=1e-4):
    """
    Symmetric normalization: D^{-1/2} A D^{-1/2} [cite: 162-164].
    Fully vectorized and mathematically safe against gradient explosions.
    """
    # 1. Calculate degree across the batch
    # Shape: [B, N]
    degree = adj.sum(dim=-1) 
    
    # 2. Initialize a safe degree tensor with 1.0s to prevent infinite gradients
    safe_degree = torch.ones_like(degree)
    
    # 3. Identify valid nodes (degree must be substantially above 0 to be safe)
    # If node_mask is provided, combine it with the degree check
    valid_nodes = degree > threshold
    if node_mask is not None:
        valid_nodes = valid_nodes & node_mask
        
    # 4. Copy actual degrees only for valid nodes
    safe_degree[valid_nodes] = degree[valid_nodes]
    
    # 5. Safe Power Operation
    # Valid nodes get x^-0.5. Empty nodes get 1.0^-0.5 = 1.0 (Gradient is safe!)
    d_inv_sqrt = safe_degree ** -0.5
    
    # 6. Zero out the empty/padded nodes AFTER the power operation
    d_inv_sqrt = d_inv_sqrt * valid_nodes.float()
    
    # 7. Create diagonal matrices for the whole batch simultaneously
    # Shape: [B, N, N]
    D_inv_sqrt_mat = torch.diag_embed(d_inv_sqrt)
    
    # 8. Batched matrix multiplication
    return torch.bmm(D_inv_sqrt_mat, torch.bmm(adj, D_inv_sqrt_mat))

def _linear_ramp(epoch, warmup_epochs):
    """Return a multiplier that linearly ramps from 0 → 1 over `warmup_epochs`."""
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, epoch / warmup_epochs)

def _sigmoid(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, 1 / (1 + exp((-epoch + warmup_epochs) / 10)))


# ══════════════════════════════════════════════════════════════
# Individual Loss Functions  (operate on a single layer result dict)
# ══════════════════════════════════════════════════════════════

# ---- MoINN losses (type-level) --------------------------------

def cut_loss(result):
    A = normalize(result["adjacency_min"])
    C = result["distance_dependent_type_similarity_soft"]
    num = batch_trace(C.transpose(1, 2) @ A @ C)
    D = batch_diag(A.sum(dim=-1))
    den = batch_trace(C.transpose(1, 2) @ D @ C).clamp(min=1e-8)
    return -num / den

def ortho_loss(result):
    SS_t = result["type_similarity"]
    N = SS_t.shape[1]
    I = torch.eye(N, device=SS_t.device)
    norm = batch_norm(SS_t).clamp(min=1e-8)
    return batch_norm(batch_div(SS_t, norm) - I / torch.norm(I)).clamp(min=1e-8)

def entropy_loss(result):
    S = result["type_assignments"]
    EPS = 1e-8
    return -(S * torch.log(S + EPS)).sum(dim=-1).mean(dim=-1)

# ---- Pooling losses (spatial-level) ----------------------------

def recon_loss(result):
    C_soft = normalize(result["softbfs"].detach())
    N = C_soft.shape[1]
    S_pool = result["spatial_assignments"]
    return (torch.sum((S_pool @ S_pool.transpose(1, 2) - C_soft)**2, dim=(1, 2)).mean() / N**2)

def cut_loss_pool(result):
    A_mol = result["adjacency_mol"]
    S_pool = result["spatial_assignments"]
    A = normalize(A_mol)
    num = batch_trace(S_pool.transpose(1, 2) @ A @ S_pool)
    D = batch_diag(A.sum(dim=-1))
    den = batch_trace(S_pool.transpose(1, 2) @ D @ S_pool).clamp(min=1e-12)
    return -num / den

def ortho_loss_pool(result):
    S = result["spatial_assignments"]
    M = S.shape[-1]
    I = torch.eye(M, device=S.device)
    StS = S.transpose(1, 2) @ S
    return batch_norm(batch_div(StS, batch_norm(StS).clamp(min=1e-12)) - I / torch.norm(I))

def link_loss_pool(result):
    A_mol = result["distance_dependent_type_similarity_hard"]
    B, N, _ = A_mol.shape
    S_pool = result["spatial_assignments"]
    return (torch.sum((S_pool @ S_pool.transpose(1, 2)) *
            (torch.ones(N, N, device=A_mol.device).unsqueeze(0).expand(B, -1, -1) - A_mol),
            dim=(1, 2)).mean() / N**2)

def entropy_loss_pool(result):
    S = result["spatial_assignments"]
    EPS = 1e-15
    return -(S * torch.log(S + EPS)).sum(dim=-1).mean(dim=-1)


# ══════════════════════════════════════════════════════════════
# Per-layer & Total Loss
# ══════════════════════════════════════════════════════════════

def layer_total_loss(result, epoch, cfg: LayerLossConfig):
    """Compute all loss terms for a single layer."""
    ZERO = torch.tensor(0.0, device=result["type_assignments"].device)

    # MoINN losses (type-level, always active)
    cut_w   = cfg.w_cut_moinn   * _sigmoid(epoch, cfg.warmup_cut_moinn)
    ortho_w = cfg.w_ortho_moinn *_sigmoid(epoch, cfg.warmup_ortho_moinn)
    ent_w   = cfg.w_ent_moinn   * _sigmoid(epoch, cfg.warmup_ent_moinn)

    l_cut   = cut_loss(result)     * cut_w   if cut_w   > 0 else ZERO
    l_ortho = ortho_loss(result)   * ortho_w if ortho_w > 0 else ZERO
    l_ent   = entropy_loss(result) * ent_w   if ent_w   > 0 else ZERO

    # Pooling losses (spatial-level, delayed start + warmup)
    if epoch <= cfg.pool_start_epoch:
        l_recon, l_cut_p, l_ortho_p, l_link_p, l_ent_p = ZERO, ZERO, ZERO, ZERO, ZERO
    else:
        pe = epoch - cfg.pool_start_epoch
        recon_eff  = cfg.w_recon_pool * _sigmoid(pe, cfg.warmup_recon_pool)
        cut_p_eff  = cfg.w_cut_pool   * _sigmoid(pe, cfg.warmup_cut_pool)
        ortho_p_eff = cfg.w_ortho_pool
        link_p_eff = cfg.w_link_pool  * _sigmoid(pe, cfg.warmup_link_pool)
        ent_p_eff  = cfg.w_ent_pool   * _sigmoid(pe, cfg.warmup_ent_pool)

        l_recon   = recon_loss(result)      * recon_eff   if recon_eff   > 0 else ZERO
        l_cut_p   = cut_loss_pool(result)   * cut_p_eff   if cut_p_eff   > 0 else ZERO
        l_ortho_p = ortho_loss_pool(result) * ortho_p_eff if ortho_p_eff > 0 else ZERO
        l_link_p  = link_loss_pool(result)  * link_p_eff  if link_p_eff  > 0 else ZERO
        l_ent_p   = entropy_loss_pool(result) * ent_p_eff if ent_p_eff   > 0 else ZERO

    total = torch.mean(l_cut + l_ortho + l_ent + l_recon + l_cut_p + l_ortho_p + l_link_p + l_ent_p)
    return (
        total,
        torch.mean(l_cut), torch.mean(l_ortho), torch.mean(l_ent),
        torch.mean(l_recon), torch.mean(l_cut_p), torch.mean(l_ortho_p),
        torch.mean(l_link_p), torch.mean(l_ent_p),
    )


def total_loss(layer_results, epoch, cfg: HierarchicalLossConfig):
    """Compute total loss summed across all layers.

    Returns:
        (grand_total, all_terms)
        where all_terms is a list of 9-tuples per layer.
    """
    dev = layer_results[0]["type_assignments"].device
    grand_total = torch.tensor(0.0, device=dev)
    all_terms = []
    for result, layer_cfg in zip(layer_results, cfg.layer_configs):
        terms = layer_total_loss(result, epoch, layer_cfg)
        grand_total = grand_total + terms[0]
        all_terms.append(terms)
    return grand_total, all_terms

In [12]:
# ══════════════════════════════════════════════════════════════
# Quick Smoke Test
# ══════════════════════════════════════════════════════════════
print("--- Testing Model Components ---")

N_LAYERS = 2
test_model = HierarchicalMincutTopoModel(
    n_layers=N_LAYERS,
    hidden_channels=16,
    n_filters=64,
    n_interactions=1,
    n_clusters=10,
    schnet_cutoff=10.0,
    r_min_cutoff=2.0,
    r_dist_cutoff=3.5,
    max_beads=8,
    layer_configs=[
        {},                                   # Layer 0: defaults
        {'max_beads': 4, 'n_clusters': 5},    # Layer 1: fewer beads
    ]
).to(device)

test_cfg = HierarchicalLossConfig(n_layers=N_LAYERS)
print("Model initialized.")

try:
    batch = next(iter(train_loader))
    batch = batch.to(device)
    z, pos, adj, mask = batch.z, batch.pos, batch.adj, batch.mask
    print(f"Input shapes: z={z.shape}, pos={pos.shape}, adj={adj.shape}, mask={mask.shape}")

    test_model.eval()
    with torch.no_grad():
        results = test_model(z, pos, adj, mask=mask)

    for i, r in enumerate(results):
        print(f"\nLayer {i} output shapes:")
        for k, v in r.items():
            print(f"  {k}: {v.shape}")

    loss_val, all_terms = total_loss(results, epoch=0, cfg=test_cfg)
    print(f"\nGrand Total Loss: {loss_val.item():.4f}")
    for i, terms in enumerate(all_terms):
        print(f"  Layer {i}: Total={terms[0].item():.4f}  Cut={terms[1].item():.4f} "
              f"Ortho={terms[2].item():.4f}  Ent={terms[3].item():.4f}")

    _, all_moieties = test_model.get_hard_moieties(z, pos, adj, mask=mask)
    for i, moieties in enumerate(all_moieties):
        print(f"  Layer {i} moieties (sample 0): {moieties[0]}")

    print("\n--- Test Complete ---")

except Exception as e:
    print(f"\n!!! Test Failed: {e} !!!")
    import traceback
    traceback.print_exc()

--- Testing Model Components ---
Model initialized.
Input shapes: z=torch.Size([32, 29]), pos=torch.Size([32, 29, 3]), adj=torch.Size([32, 29, 29]), mask=torch.Size([32, 29])

Layer 0 output shapes:
  original_node_features: torch.Size([32, 29, 16])
  type_assignments: torch.Size([32, 29, 10])
  type_assignments_hard: torch.Size([32, 29, 10])
  spatial_assignments: torch.Size([32, 29, 8])
  spatial_assignments_hard: torch.Size([32, 29, 8])
  adjacency_dist: torch.Size([32, 29, 29])
  adjacency_min: torch.Size([32, 29, 29])
  adjacency_mol: torch.Size([32, 29, 29])
  coarse_node_features: torch.Size([32, 8, 16])
  coarse_adjacency: torch.Size([32, 8, 8])
  coarse_adjacency_hard: torch.Size([32, 8, 8])
  cluster_com: torch.Size([32, 8, 3])
  cluster_mask: torch.Size([32, 8])
  type_similarity: torch.Size([32, 29, 29])
  atom_mask: torch.Size([32, 29])
  distance_dependent_type_similarity_soft: torch.Size([32, 29, 29])
  distance_dependent_type_similarity_hard: torch.Size([32, 29, 29])
  

In [ ]:
# =============================================================
# VISUALIZATION: All visualization helpers and plotting functions
# =============================================================

ATOMIC_SYMBOLS = {1: 'H', 6: 'C', 7: 'N', 8: 'O', 9: 'F', 16: 'S', 17: 'Cl', 35: 'Br'}
ELEMENT_COLORS = {
    'H': '#AAAAAA', 'C': '#35CFD4', 'N': '#3050F8',
    'O': '#FF0D0D', 'F': '#90E050', 'S': '#FFFF30',
    'Cl': '#1FF01F', 'Br': '#A62929'
}


def _sym(atomic_num):
    return ATOMIC_SYMBOLS.get(int(atomic_num), f'Z{int(atomic_num)}')


def _make_atom_labels(z):
    symbols = [_sym(a) for a in z]
    counts = Counter(symbols)
    running = {}
    labels = []
    for s in symbols:
        if counts[s] > 1:
            running[s] = running.get(s, 0) + 1
            labels.append(f'{s}{running[s]}')
        else:
            labels.append(s)
    return labels


def _make_full_labels(atom_labels, mask):
    if atom_labels is None or mask is None:
        return None
    mask_np = mask.detach().cpu().numpy().astype(bool) if torch.is_tensor(mask) else np.asarray(mask, dtype=bool)
    full = []
    valid_idx = 0
    for m in mask_np:
        if m:
            full.append(atom_labels[valid_idx])
            valid_idx += 1
        else:
            full.append('0')
    return full


def _element_node_colors(z, alpha=1.0):
    return [to_rgba(ELEMENT_COLORS.get(_sym(z[i]), '#FF69B4'), alpha=alpha) for i in range(len(z))]


def _cluster_node_colors(labels, alpha=1.0):
    cmap = plt.get_cmap('tab20')
    return [to_rgba(cmap(int(l) % 20), alpha=alpha) for l in labels]


def _cluster_color_map(labels, cmap_name='tab20'):
    cmap = plt.get_cmap(cmap_name)
    uniq = [int(u) for u in np.unique(np.asarray(labels, dtype=int))]
    return {u: cmap(u % 20) for u in uniq}


def _build_nx_graph(N, adj_np):
    G = nx.Graph()
    G.add_nodes_from(range(N))
    rows, cols = np.where(adj_np > 0)
    for i, j in zip(rows, cols):
        if i < j:
            G.add_edge(int(i), int(j))
    return G


def _mol_layout(G, pos_3d):
    init = {i: (pos_3d[i, 0], pos_3d[i, 1]) for i in range(len(pos_3d))}
    return nx.spring_layout(G, pos=init, seed=42, iterations=80)


def _edge_labels_from_dists(G, edge_dists_np):
    labels = {}
    for (i, j) in G.edges():
        d = edge_dists_np[i, j] if edge_dists_np[i, j] > 0 else edge_dists_np[j, i]
        if d > 0:
            labels[(i, j)] = f'{d:.2f}'
    return labels


def _draw_graph(ax, G, layout, label_dict, node_colors, title,
                edge_color='#888888', edge_width=1.5, node_size=350,
                edge_labels=None):
    nx.draw_networkx_edges(G, layout, ax=ax, edge_color=edge_color, width=edge_width)
    nx.draw_networkx_nodes(
        G, layout, ax=ax, node_color=node_colors,
        node_size=node_size, edgecolors='black', linewidths=0.8
    )
    nx.draw_networkx_labels(G, layout, labels=label_dict, ax=ax, font_size=7, font_weight='bold')
    if edge_labels:
        nx.draw_networkx_edge_labels(
            G, layout, edge_labels=edge_labels, ax=ax,
            font_size=5, font_color='#444444',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.7)
        )
    ax.set_title(title, fontsize=13)
    ax.axis('off')


def _nn_scale(points):
    p = np.asarray(points, dtype=float)
    if len(p) <= 1:
        return 0.10
    d = p[:, None, :] - p[None, :, :]
    d = np.sqrt((d ** 2).sum(axis=-1))
    d += np.eye(len(p)) * 1e6
    nn = d.min(axis=1)
    return float(np.clip(np.median(nn), 0.05, 0.45))


def _draw_dashed_cluster_loops(ax, layout, labels, color_map=None, pad=0.05, lw=2.1, alpha=0.95):
    """Draw smooth molecule-following loops using density contours around nodes."""
    labels = np.asarray(labels, dtype=int)
    if color_map is None:
        color_map = _cluster_color_map(labels)

    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        if idx.size == 0:
            continue

        pts = np.array([layout[int(i)] for i in idx], dtype=float)
        col = color_map[int(c)]

        s = _nn_scale(pts)
        sigma = float(np.clip(0.42 * s, 0.035, 0.16))
        target_r = float(np.clip(0.45 * s + pad, 0.055, 0.20))

        min_xy = pts.min(axis=0) - (target_r + 0.06)
        max_xy = pts.max(axis=0) + (target_r + 0.06)
        n_grid = int(np.clip(120 + 30 * len(pts), 120, 260))

        xs = np.linspace(min_xy[0], max_xy[0], n_grid)
        ys = np.linspace(min_xy[1], max_xy[1], n_grid)
        X, Y = np.meshgrid(xs, ys)

        Z = np.zeros_like(X)
        inv = 1.0 / (2.0 * sigma * sigma)
        for p in pts:
            Z += np.exp(-((X - p[0]) ** 2 + (Y - p[1]) ** 2) * inv)

        lvl_rel = float(np.exp(-(target_r ** 2) * inv))
        lvl_rel = float(np.clip(lvl_rel, 0.22, 0.78))
        level = lvl_rel * float(Z.max())

        ax.contour(
            X, Y, Z,
            levels=[level],
            colors=[col],
            linewidths=lw,
            linestyles='--',
            alpha=alpha,
        )

    return color_map


def _atoms_to_layer_inputs(layer_spatial_labels):
    """Return atom->input-node index mapping for each layer."""
    maps = []
    if not layer_spatial_labels:
        return maps

    atom_to_cur = np.arange(len(layer_spatial_labels[0]), dtype=int)
    maps.append(atom_to_cur.copy())
    for i in range(1, len(layer_spatial_labels)):
        prev_assign = np.asarray(layer_spatial_labels[i - 1], dtype=int)
        atom_to_cur = prev_assign[atom_to_cur]
        maps.append(atom_to_cur.copy())
    return maps


# --------------------------------------------------------------
# Multi-layer heatmap visualization
# --------------------------------------------------------------
def viz_heatmaps_multilayer(layer_results, layer_masks, epoch, save_dir, layer_atom_labels=None):
    n_layers = len(layer_results)

    layer_full_labels = []
    for i in range(n_layers):
        if layer_atom_labels and layer_atom_labels[i] is not None:
            layer_full_labels.append(_make_full_labels(layer_atom_labels[i], layer_masks[i]))
        else:
            layer_full_labels.append(None)

    configs = [
        ('s_space_heatmap', 'viridis', 'Spatial Assignments (S_space)'),
        ('adj_heatmap', 'inferno', 'Coarse Adjacency'),
        ('c_h_heatmap', 'magma', 'Hard Moiety Matrix (C_h)'),
        ('s_type_heatmap', 'cividis', 'Type Assignments (S_type)'),
        ('c_soft_heatmap', 'plasma', 'Dist-Dep Type Similarity (C_soft)'),
        ('c_prime_heatmap', 'coolwarm', 'Soft BFS (C_prime)'),
    ]

    all_paths = {}
    for dir_name, cmap_name, base_title in configs:
        d = os.path.join(save_dir, dir_name)
        os.makedirs(d, exist_ok=True)

        fig, axes = plt.subplots(1, n_layers, figsize=(10 * n_layers, 8), squeeze=False)
        for i in range(n_layers):
            m = layer_masks[i]
            r = layer_results[i]
            n_v = int(m.sum())
            ax = axes[0, i]

            if dir_name == 's_space_heatmap':
                data = r['spatial_assignments'][0][:n_v].detach().cpu().numpy()
                y_lbl = layer_atom_labels[i] if layer_atom_labels else True
                x_lbl = True
            elif dir_name == 'adj_heatmap':
                data = r['coarse_adjacency_hard'][0].detach().cpu().numpy()
                y_lbl, x_lbl = True, True
            elif dir_name == 'c_h_heatmap':
                data = r['distance_dependent_type_similarity_hard'][0].detach().cpu().numpy()
                y_lbl = layer_full_labels[i] if layer_full_labels[i] else True
                x_lbl = y_lbl
            elif dir_name == 's_type_heatmap':
                data = r['type_assignments'][0][:n_v].detach().cpu().numpy()
                y_lbl = layer_atom_labels[i] if layer_atom_labels else True
                x_lbl = True
            elif dir_name == 'c_soft_heatmap':
                t = r['distance_dependent_type_similarity_soft'][0]
                data = t[m][:, m].detach().cpu().numpy()
                y_lbl = layer_atom_labels[i] if layer_atom_labels else True
                x_lbl = y_lbl
            else:
                t = r['softbfs'][0]
                data = t[m][:, m].detach().cpu().numpy()
                y_lbl = layer_atom_labels[i] if layer_atom_labels else True
                x_lbl = y_lbl

            sns.heatmap(
                data, cmap=cmap_name, cbar=True, ax=ax,
                yticklabels=y_lbl if y_lbl else True,
                xticklabels=x_lbl if x_lbl else True
            )
            ax.set_title(f'{base_title} - Layer {i} - Epoch {epoch}', fontsize=13)
            if isinstance(y_lbl, list):
                ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
            if isinstance(x_lbl, list):
                ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=90)

        plt.tight_layout()
        p = os.path.join(d, f'epoch_{epoch}.png')
        plt.savefig(p, bbox_inches='tight')
        plt.close(fig)
        all_paths[dir_name] = p

    return all_paths


# --------------------------------------------------------------
# Node features heatmap (layer 0 only)
# --------------------------------------------------------------
def viz_node_features(node_features, mask, epoch, save_dir, atom_labels=None):
    feat_dir = os.path.join(save_dir, 'node_feat_heatmap')
    os.makedirs(feat_dir, exist_ok=True)

    feat_np = node_features.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy().astype(bool)
    feat_valid = feat_np[mask_np]

    fig = plt.figure(figsize=(max(12, feat_valid.shape[1] * 0.25), max(6, feat_valid.shape[0] * 0.4)))
    ax = sns.heatmap(
        feat_valid,
        cmap='viridis',
        cbar=True,
        xticklabels=True,
        yticklabels=atom_labels if atom_labels else True
    )
    ax.set_xlabel('Feature dimension')
    ax.set_ylabel('Atom')
    plt.title(f'Node Features - Epoch {epoch}')
    plt.tight_layout()

    p = os.path.join(feat_dir, f'epoch_{epoch}.png')
    plt.savefig(p, dpi=150)
    plt.close(fig)
    return p


# --------------------------------------------------------------
# Cluster visualization (hierarchical)
# --------------------------------------------------------------
def save_cluster_viz_multilayer(layer_graph_data, layer_moieties, layer_spatial_labels, epoch, save_dir):
    n_layers = len(layer_graph_data)
    z0, pos0, adj0, edge_dists0 = layer_graph_data[0]
    N0 = len(z0)

    G0 = _build_nx_graph(N0, adj0)
    layout0 = _mol_layout(G0, pos0)
    labels0 = _make_atom_labels(z0)
    label_dict0 = {j: l for j, l in enumerate(labels0)}
    edge_labels0 = _edge_labels_from_dists(G0, edge_dists0) if edge_dists0 is not None else None

    atom_moiety_colors = _cluster_node_colors(layer_moieties[0])
    atom_spatial_colors = _cluster_node_colors(layer_spatial_labels[0])
    atom_to_layer_input = _atoms_to_layer_inputs(layer_spatial_labels)

    fig, axes = plt.subplots(2, n_layers, figsize=(8 * n_layers, 14), squeeze=False)

    for i in range(n_layers):
        if i == 0:
            _draw_graph(
                axes[0, i], G0, layout0, label_dict0, atom_moiety_colors,
                f'Moiety Clusters - Layer {i} - Epoch {epoch}', edge_labels=edge_labels0
            )
            _draw_graph(
                axes[1, i], G0, layout0, label_dict0, atom_spatial_colors,
                f'Spatial Clusters - Layer {i} - Epoch {epoch}', edge_labels=edge_labels0
            )
            continue

        _draw_graph(
            axes[0, i], G0, layout0, label_dict0, atom_moiety_colors,
            f'Moiety Groups on Atoms - Layer {i} - Epoch {epoch}', edge_labels=edge_labels0
        )
        _draw_graph(
            axes[1, i], G0, layout0, label_dict0, atom_spatial_colors,
            f'Spatial Groups on Atoms - Layer {i} - Epoch {epoch}', edge_labels=edge_labels0
        )

        map_i = atom_to_layer_input[i]

        moiety_i = np.asarray(layer_moieties[i], dtype=int)
        moiety_on_atoms = moiety_i[map_i]
        _draw_dashed_cluster_loops(
            axes[0, i], layout0, moiety_on_atoms,
            color_map=_cluster_color_map(moiety_on_atoms), pad=0.04
        )

        spatial_i = np.asarray(layer_spatial_labels[i], dtype=int)
        spatial_on_atoms = spatial_i[map_i]
        _draw_dashed_cluster_loops(
            axes[1, i], layout0, spatial_on_atoms,
            color_map=_cluster_color_map(spatial_on_atoms), pad=0.04
        )

    plt.tight_layout()
    fp = os.path.join(save_dir, f'epoch_{epoch}.png')
    plt.savefig(fp, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return fp


# --------------------------------------------------------------
# Mol vs C_h visualization
# --------------------------------------------------------------
def save_mol_vs_c_h_multilayer(layer_graph_data, layer_c_h_valid, epoch, save_dir):
    viz_dir = os.path.join(save_dir, 'mol_vs_c_h')
    os.makedirs(viz_dir, exist_ok=True)

    n_layers = len(layer_graph_data)
    fig, axes = plt.subplots(1, 2 * n_layers, figsize=(8 * n_layers, 7), squeeze=False)

    for i, (z, pos, adj, edge_dists) in enumerate(layer_graph_data):
        N = len(z)
        labels = _make_atom_labels(z) if i == 0 else [f'B{j}' for j in range(N)]
        label_dict = {j: l for j, l in enumerate(labels)}

        G_mol = _build_nx_graph(N, adj)
        G_ch = _build_nx_graph(N, layer_c_h_valid[i])
        layout = _mol_layout(G_mol, pos)
        colors = _element_node_colors(z) if i == 0 else _cluster_node_colors(range(N))
        el = _edge_labels_from_dists(G_mol, edge_dists) if edge_dists is not None else None

        _draw_graph(axes[0, 2 * i], G_mol, layout, label_dict, colors, f'Input Graph - Layer {i} - Epoch {epoch}', edge_labels=el)
        _draw_graph(
            axes[0, 2 * i + 1], G_ch, layout, label_dict, colors,
            f'C_h Connectivity - Layer {i} - Epoch {epoch}', edge_color='#CC4444'
        )

    plt.tight_layout()
    fp = os.path.join(viz_dir, f'epoch_{epoch}.png')
    plt.savefig(fp, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return fp


# --------------------------------------------------------------
# Coarse-grained final visualization (hierarchical)
# --------------------------------------------------------------
def save_coarse_grained_viz_multilayer(layer_graph_data, layer_spatial_labels, layer_cluster_com, layer_coarse_adj, save_dir):
    n_layers = len(layer_graph_data)
    fig, axes = plt.subplots(1, 2 * n_layers, figsize=(8 * n_layers, 7), squeeze=False)

    for i in range(n_layers):
        z, pos, adj, edge_dists = layer_graph_data[i]
        N = len(z)

        G_in = _build_nx_graph(N, adj)
        layout_in = _mol_layout(G_in, pos)
        in_labels = _make_atom_labels(z) if i == 0 else [f'B{j}' for j in range(N)]
        in_label_dict = {j: l for j, l in enumerate(in_labels)}

        spatial_i = np.asarray(layer_spatial_labels[i], dtype=int)
        spatial_i_colors = _cluster_node_colors(spatial_i)
        edge_labels = _edge_labels_from_dists(G_in, edge_dists) if edge_dists is not None else None

        _draw_graph(
            axes[0, 2 * i], G_in, layout_in, in_label_dict, spatial_i_colors,
            f'Input / Spatial - Layer {i}', edge_labels=edge_labels
        )

        loop_color_map = _cluster_color_map(spatial_i)
        if i + 1 < n_layers:
            next_labels = np.asarray(layer_spatial_labels[i + 1], dtype=int)
            next_on_input = next_labels[spatial_i]
            loop_color_map = _cluster_color_map(next_on_input)
            _draw_dashed_cluster_loops(
                axes[0, 2 * i], layout_in, next_on_input,
                color_map=loop_color_map, pad=0.05, lw=2.3
            )
            axes[0, 2 * i].set_title(f'Input + Layer {i + 1} Pooling Loops')

        cluster_com_np = layer_cluster_com[i]
        coarse_adj_np = layer_coarse_adj[i]
        K = cluster_com_np.shape[0]
        active = [k for k in range(K) if np.abs(cluster_com_np[k]).sum() > 1e-4]
        old2new = {k: idx for idx, k in enumerate(active)}

        G_cg = nx.Graph()
        G_cg.add_nodes_from(range(len(active)))
        for k1 in active:
            for k2 in active:
                if k1 < k2 and coarse_adj_np[k1, k2] > 0.05:
                    G_cg.add_edge(old2new[k1], old2new[k2])

        if len(active) > 0:
            cg_layout = {}
            mol_coords = np.array(list(layout_in.values()))
            for k in active:
                member_indices = [n for n in range(N) if int(spatial_i[n]) == int(k)]
                if member_indices:
                    centroid = np.mean([np.array(layout_in[m]) for m in member_indices], axis=0)
                    cg_layout[old2new[k]] = (centroid[0], centroid[1])
                else:
                    cg_layout[old2new[k]] = tuple(mol_coords.mean(axis=0))

            cg_labels = {old2new[k]: f'B{old2new[k]}' for k in active}

            cg_node_colors = []
            if i + 1 < n_layers:
                next_labels = np.asarray(layer_spatial_labels[i + 1], dtype=int)
                for k in active:
                    gid = int(next_labels[int(k)]) if int(k) < len(next_labels) else int(k)
                    cg_node_colors.append(loop_color_map.get(gid, plt.get_cmap('tab20')(gid % 20)))
            else:
                for k in active:
                    cg_node_colors.append(loop_color_map.get(int(k), plt.get_cmap('tab20')(int(k) % 20)))

            cg_edge_labels = {}
            for (n1, n2) in G_cg.edges():
                p1 = np.array(cluster_com_np[active[n1]])
                p2 = np.array(cluster_com_np[active[n2]])
                cg_edge_labels[(n1, n2)] = f'{np.linalg.norm(p1 - p2):.2f}'

            _draw_graph(
                axes[0, 2 * i + 1], G_cg, cg_layout, cg_labels, cg_node_colors,
                f'CG Graph - Layer {i}', node_size=500, edge_labels=cg_edge_labels
            )
        else:
            axes[0, 2 * i + 1].set_title(f'CG Graph - Layer {i} (empty)')
            axes[0, 2 * i + 1].axis('off')

    plt.tight_layout()
    fp = os.path.join(save_dir, 'final_coarse_viz.png')
    plt.savefig(fp, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved coarse grained visualization to {fp}')
    return fp


# --------------------------------------------------------------
# GIF helper
# --------------------------------------------------------------
def save_gif(paths, output_path, fps=1, max_width=800):
    if not paths:
        return
    try:
        writer = imageio.get_writer(output_path, mode='I', fps=fps)
        for p in paths:
            img = _PILImage.open(p)
            if img.width > max_width:
                ratio = max_width / img.width
                img = img.resize((max_width, int(img.height * ratio)), _PILImage.LANCZOS)
            frame = np.asarray(img.convert('RGB'))
            writer.append_data(frame)
            img.close()
            del frame, img
        writer.close()
        gc.collect()
        print(f'GIF saved as {output_path}')
    except Exception as e:
        print(f'Failed to save GIF {output_path}: {e}')

In [37]:
import matplotlib
matplotlib.use('Agg')

# ──────────────────────────────────────────────────────────────
# Graceful interrupt handler
# ──────────────────────────────────────────────────────────────
_stop_training = False

def _sigint_handler(signum, frame):
    global _stop_training
    if _stop_training:
        raise KeyboardInterrupt
    _stop_training = True
    print("\n>>> Interrupt received. Will stop after current batch finishes...")

# ══════════════════════════════════════════════════════════════
# Training Configuration
# ══════════════════════════════════════════════════════════════
LEARNING_RATE = 3e-4
EPOCHS = 250
N_LAYERS = 2

# Gradient logging configuration
GRAD_LOG_EVERY_EPOCH = 1
GRAD_LOG_FIRST_BATCH_ONLY = True

def _grad_norm_or_none(t):
    if t is None or not torch.is_tensor(t):
        return None
    tt = t.detach()
    if not tt.is_floating_point() and not tt.is_complex():
        tt = tt.float()
    return float(torch.linalg.norm(tt).item())

def _capture_grad_snapshot(model, layer_results):
    snap = {}

    # Activation gradients per hierarchical layer
    act_keys = [
        'original_node_features',
        'type_assignments',
        'distance_dependent_type_similarity_soft',
        'distance_dependent_type_similarity_hard',
        'softbfs',
        'spatial_assignments',
        'coarse_node_features',
        'coarse_adjacency',
    ]
    for i, result in enumerate(layer_results):
        for k in act_keys:
            t = result.get(k, None)
            snap[f'L{i}::act::{k}'] = _grad_norm_or_none(t.grad if torch.is_tensor(t) else None)

    # Parameter gradients per hierarchical layer (grouped)
    param_groups_local = [
        'encoder.embedding',
        'encoder.interactions',
        'type_mapping',
        'softBFS',
        'coarse_grain_layer',
    ]
    named_params = list(model.named_parameters())
    for i in range(len(model.layers)):
        prefix = f'layers.{i}.'
        for g in param_groups_local:
            vals = [
                _grad_norm_or_none(p.grad) for n, p in named_params
                if n.startswith(prefix) and (g in n) and p.grad is not None
            ]
            vals = [v for v in vals if v is not None]
            if len(vals) == 0:
                snap[f'L{i}::param::{g}'] = None
            else:
                snap[f'L{i}::param::{g}'] = float(np.mean(vals))

    return snap

# ── Per-layer loss config (set coefficients & warmups individually) ──
LOSS_CFG = HierarchicalLossConfig(
    n_layers=N_LAYERS,
    layer_configs=[
        LayerLossConfig(
            w_cut_moinn=1.0, w_ortho_moinn=1.0, w_ent_moinn=0.16,
            warmup_cut_moinn=10, warmup_ortho_moinn=0, warmup_ent_moinn=20,
            w_recon_pool=0.0, w_cut_pool=0.0, w_ortho_pool=0.0,
            w_link_pool=0.0, w_ent_pool=0.0,
            pool_start_epoch=0,
        ),
        LayerLossConfig(
            w_cut_moinn=1.0, w_ortho_moinn=1.0, w_ent_moinn=0.16,
            warmup_cut_moinn=30, warmup_ortho_moinn=10, warmup_ent_moinn=40,
            w_recon_pool=0.0, w_cut_pool=0.0, w_ortho_pool=0.0,
            w_link_pool=0.0, w_ent_pool=0.0,
            pool_start_epoch=40,
        ),
    ]
)

# ── Model ──
model = HierarchicalMincutTopoModel(
    n_layers=N_LAYERS,
    hidden_channels=128,
    n_filters=64,
    n_interactions=3,
    n_clusters=100,
    schnet_cutoff=10.0,
    r_min_cutoff=2.0,
    r_dist_cutoff=3.5,
    max_beads=16,
    layer_configs=[
        {},                                                        # Layer 0: defaults
        {'max_beads': 8, 'n_clusters': 10, 'schnet_cutoff': 10.0,
         'r_min_cutoff': 2.5, 'r_dist_cutoff': 4.0, 'temperature': 10.0, 'max_iters': 3},              # Layer 1: adjusted for CG beads
    ]
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model.reset_parameters()
print("Model parameters reset.")

n_layers = len(model.layers)

# ── Pre-training GPU cleanup ──
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
plt.close('all'); gc.collect()
if torch.cuda.is_available():
    print(f"GPU memory before training: {torch.cuda.memory_allocated()/1024**2:.1f} MB allocated, "
          f"{torch.cuda.memory_reserved()/1024**2:.1f} MB reserved")
else:
    print("CPU mode")

# ── Visualisation sample (fixed across epochs) ──
_first_batch = next(iter(train_loader))
_first_batch = _first_batch.to(device)
viz_z    = _first_batch.z[0:1]
viz_pos  = _first_batch.pos[0:1]
viz_adj  = _first_batch.adj[0:1]
viz_mask = _first_batch.mask[0:1]

viz_z_0   = viz_z[0][viz_mask[0]].cpu().numpy()
viz_pos_0 = viz_pos[0][viz_mask[0]].cpu().numpy()
viz_adj_0 = viz_adj[0][viz_mask[0]][:, viz_mask[0]].cpu().numpy()
viz_edge_dists_0 = _first_batch.edge_dists[0][viz_mask[0]][:, viz_mask[0]].cpu().numpy()
viz_atom_labels = _make_atom_labels(viz_z_0)
del _first_batch

# ── Path accumulators  ──
image_paths, mol_vs_c_h_paths, node_feat_heatmap_paths = [], [], []
heatmap_path_histories = {
    's_space_heatmap': [], 'adj_heatmap': [], 'c_h_heatmap': [],
    's_type_heatmap': [], 'c_soft_heatmap': [], 'c_prime_heatmap': [],
}

# ── Loss histories ──
loss_names = ['total', 'cut', 'ortho', 'ent', 'recon', 'cut_pool', 'ortho_pool', 'link_pool', 'ent_pool']
loss_history = []  # grand total
layer_loss_histories = [{name: [] for name in loss_names} for _ in range(n_layers)]
grad_history = []

# ── Cleanup old visualisations ──
print(f"Cleaning previously saved visualizations in {SAVE_DIR}...")
if os.path.exists(SAVE_DIR):
    for fname in os.listdir(SAVE_DIR):
        fp = os.path.join(SAVE_DIR, fname)
        try:
            if os.path.isfile(fp) and fname.endswith(('.png', '.gif')):
                os.unlink(fp)
            elif os.path.isdir(fp) and fname in (
                's_space_heatmap', 'adj_heatmap', 'c_h_heatmap',
                's_type_heatmap', 'c_soft_heatmap', 'c_prime_heatmap',
                'mol_vs_c_h', 'node_feat_heatmap'
            ):
                shutil.rmtree(fp)
        except Exception as e:
            print(f'Failed to delete {fp}: {e}')

# ── Install signal handler ──
_stop_training = False
_prev_handler = signal.signal(signal.SIGINT, _sigint_handler)

print("Starting training... (press Ctrl+C once to stop gracefully, twice to force)")

for epoch in range(EPOCHS):
    if _stop_training:
        break

    model.train()
    running_total = 0.0
    running_layer = [{name: 0.0 for name in loss_names} for _ in range(n_layers)]
    grad_epoch = {}

    for step_idx, batch in enumerate(train_loader):
        if _stop_training:
            break
        batch = batch.to(device)
        optimizer.zero_grad()
        z, pos, adj, mask = batch.z, batch.pos, batch.adj, batch.mask

        results = model(z, pos, adj, mask=mask)

        tracked_for_grad = [
            'original_node_features',
            'type_assignments',
            'distance_dependent_type_similarity_soft',
            'distance_dependent_type_similarity_hard',
            'softbfs',
            'spatial_assignments',
            'coarse_node_features',
            'coarse_adjacency',
        ]
        for layer_result in results:
            for _k in tracked_for_grad:
                _t = layer_result.get(_k, None)
                if torch.is_tensor(_t) and _t.requires_grad:
                    _t.retain_grad()

        loss, all_terms = total_loss(results, epoch, cfg=LOSS_CFG)
        loss.backward()

        if (epoch % GRAD_LOG_EVERY_EPOCH == 0) and ((not GRAD_LOG_FIRST_BATCH_ONLY) or step_idx == 0):
            grad_epoch = _capture_grad_snapshot(model, results)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        

        running_total += loss.item()
        for i, terms in enumerate(all_terms):
            for j, name in enumerate(loss_names):
                running_layer[i][name] += terms[j].item()

    if _stop_training:
        print(f">>> Stopping after epoch {epoch}/{EPOCHS}.")
        break

    n_batches = len(train_loader)
    avg_total = running_total / n_batches
    loss_history.append(avg_total)
    for i in range(n_layers):
        for name in loss_names:
            layer_loss_histories[i][name].append(running_layer[i][name] / n_batches)

    print(f"Epoch {epoch+1}/{EPOCHS} | Total Loss: {avg_total:.4f}")
    for i in range(n_layers):
        avgs = {name: layer_loss_histories[i][name][-1] for name in loss_names}
        print(f"  L{i} | Cut:{avgs['cut']:.4f} Ortho:{avgs['ortho']:.4f} Ent:{avgs['ent']:.4f} "
              f"| Rec:{avgs['recon']:.4f} CutP:{avgs['cut_pool']:.4f} OrthoP:{avgs['ortho_pool']:.4f} "
              f"LinkP:{avgs['link_pool']:.4f} EntP:{avgs['ent_pool']:.4f}")

    if grad_epoch:
        grad_history.append({'epoch': epoch + 1, **grad_epoch})
        print("  Grad | Activation norms (first batch):")
        for i in range(n_layers):
            print(
                f"    L{i}: h={grad_epoch.get(f'L{i}::act::original_node_features', None)} "
                f"| s_type={grad_epoch.get(f'L{i}::act::type_assignments', None)} "
                f"| softbfs={grad_epoch.get(f'L{i}::act::softbfs', None)} "
                f"| s_space={grad_epoch.get(f'L{i}::act::spatial_assignments', None)}"
            )
        print("  Grad | Parameter mean norms:")
        for i in range(n_layers):
            print(
                f"    L{i}: enc.embed={grad_epoch.get(f'L{i}::param::encoder.embedding', None)} "
                f"| enc.inter={grad_epoch.get(f'L{i}::param::encoder.interactions', None)} "
                f"| type={grad_epoch.get(f'L{i}::param::type_mapping', None)} "
                f"| softBFS={grad_epoch.get(f'L{i}::param::softBFS', None)} "
                f"| coarse={grad_epoch.get(f'L{i}::param::coarse_grain_layer', None)}"
            )

    # ── Periodic visualisation ──
    if not _stop_training and epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            result_viz = model(viz_z, viz_pos, viz_adj, mask=viz_mask)

            # Build per-layer masks and labels
            layer_masks = [viz_mask[0]]
            for i in range(n_layers - 1):
                layer_masks.append(result_viz[i]["cg_mask"][0])

            layer_labels = [viz_atom_labels]
            for i in range(n_layers - 1):
                n_cg = int(layer_masks[i + 1].sum())
                layer_labels.append([f'B{j}' for j in range(n_cg)])

            # Heatmaps (all layers side-by-side)
            hm_paths = viz_heatmaps_multilayer(result_viz, layer_masks, epoch, SAVE_DIR, layer_labels)
            for key, path in hm_paths.items():
                heatmap_path_histories[key].append(path)

            # Node features (layer 0 only)
            node_feat_vis = result_viz[0]["original_node_features"][0]
            p = viz_node_features(node_feat_vis, viz_mask[0], epoch, SAVE_DIR,
                                  atom_labels=viz_atom_labels)
            node_feat_heatmap_paths.append(p)

            # Build per-layer graph data for cluster/graph visualisations
            layer_graph_data = [(viz_z_0, viz_pos_0, viz_adj_0, viz_edge_dists_0)]
            for i in range(n_layers - 1):
                m = result_viz[i]["cg_mask"][0]
                n_cg = int(m.sum())
                cg_z   = result_viz[i]["cg_z"][0][:n_cg].cpu().numpy()
                cg_pos = result_viz[i]["cg_pos"][0][:n_cg].cpu().numpy()
                cg_adj = result_viz[i]["cg_adj"][0][:n_cg, :n_cg].cpu().numpy()
                layer_graph_data.append((cg_z, cg_pos, cg_adj, None))

            # Get moieties & spatial labels for all layers
            _, all_moieties = model.get_hard_moieties(viz_z, viz_pos, viz_adj, mask=viz_mask)
            layer_moieties = [all_moieties[i][0] for i in range(n_layers)]

            layer_spatial_labels = []
            for i in range(n_layers):
                m = layer_masks[i]
                s_idx = torch.argmax(result_viz[i]["spatial_assignments_hard"], dim=-1)
                layer_spatial_labels.append(s_idx[0][m].cpu().numpy())

            # Cluster viz (multi-layer)
            p = save_cluster_viz_multilayer(layer_graph_data, layer_moieties,
                                            layer_spatial_labels, epoch, SAVE_DIR)
            if p:
                image_paths.append(p)

            # Mol vs C_h (multi-layer)
            layer_c_h_valid = []
            for i in range(n_layers):
                m = layer_masks[i]
                c_h = result_viz[i]["distance_dependent_type_similarity_hard"][0]
                layer_c_h_valid.append(c_h[m][:, m].cpu().numpy())
            p = save_mol_vs_c_h_multilayer(layer_graph_data, layer_c_h_valid, epoch, SAVE_DIR)
            if p:
                mol_vs_c_h_paths.append(p)

        plt.close('all'); gc.collect()

# ── Restore default signal handler ──
signal.signal(signal.SIGINT, _prev_handler)

# ── Final viz inference (BEFORE GPU cleanup) ──
_final_result_viz = None
if loss_history:
    model.eval()
    with torch.no_grad():
        _final_result_viz = model(viz_z, viz_pos, viz_adj, mask=viz_mask)
        # Store per-layer CPU tensors for final coarse-grained viz
        _final_per_layer = []
        for i in range(n_layers):
            _final_per_layer.append({
                's_hard': _final_result_viz[i]["spatial_assignments_hard"].cpu(),
                'cluster_com': _final_result_viz[i]["cluster_com"][0].cpu(),
                'coarse_adj': _final_result_viz[i]["coarse_adjacency_hard"][0].cpu(),
                'cg_mask': _final_result_viz[i]["cg_mask"][0].cpu(),
                'cg_z': _final_result_viz[i]["cg_z"][0].cpu(),
                'cg_pos': _final_result_viz[i]["cg_pos"][0].cpu(),
                'cg_adj': _final_result_viz[i]["cg_adj"][0].cpu(),
            })
    del _final_result_viz

# ── Cleanup CUDA ──
if torch.cuda.is_available():
    torch.cuda.synchronize(); torch.cuda.empty_cache()
plt.close('all'); gc.collect()

print(f"Completed {len(loss_history)} epochs. Generating outputs...")

# ── Loss curves ──
if loss_history:
    fig, axes_lc = plt.subplots(1, n_layers + 1, figsize=(8 * (n_layers + 1), 6), squeeze=False)

    # Grand total + per-layer totals
    axes_lc[0, 0].plot(loss_history, label='Grand Total', linewidth=2)
    for i in range(n_layers):
        axes_lc[0, 0].plot(layer_loss_histories[i]['total'], label=f'Layer {i}', linestyle='--')
    axes_lc[0, 0].set_title('Total Loss'); axes_lc[0, 0].legend(); axes_lc[0, 0].grid(True)
    axes_lc[0, 0].set_xlabel('Epoch'); axes_lc[0, 0].set_ylabel('Loss')

    # Per-layer breakdowns
    for i in range(n_layers):
        ax = axes_lc[0, i + 1]
        for name, ls in [('cut', '--'), ('ortho', '--'), ('ent', '--'),
                          ('recon', ':'), ('cut_pool', ':'), ('ortho_pool', ':'),
                          ('link_pool', ':'), ('ent_pool', ':')]:
            vals = layer_loss_histories[i][name]
            if any(abs(v) > 1e-12 for v in vals):
                ax.plot(vals, label=name, linestyle=ls)
        ax.set_title(f'Layer {i} Losses'); ax.legend(); ax.grid(True)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')

    plt.tight_layout()
    loss_curve_path = os.path.join(SAVE_DIR, 'loss_curves.png')
    plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"Loss curves saved to {loss_curve_path}")

# ── Gradient history curves (hierarchical) ──
if grad_history:
    def _series_from_grad_history(key):
        ys = []
        for row in grad_history:
            v = row.get(key, None)
            ys.append(np.nan if v is None else float(v))
        return np.array(ys, dtype=float)

    x = np.arange(1, len(grad_history) + 1)
    fig = plt.figure(figsize=(14, 8))
    any_plotted = False

    layer_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    for i in range(n_layers):
        c = layer_colors[i % len(layer_colors)]
        grad_traces = [
            (f'L{i}::act::original_node_features', f'L{i} Act: h', c, '-', 'o'),
            (f'L{i}::act::type_assignments', f'L{i} Act: s_type', c, '--', 's'),
            (f'L{i}::act::softbfs', f'L{i} Act: softbfs', c, '-.', 'D'),
            (f'L{i}::act::spatial_assignments', f'L{i} Act: s_space', c, ':', '^'),
            (f'L{i}::param::encoder.interactions', f'L{i} Param: enc.inter', c, '-', 'P'),
            (f'L{i}::param::type_mapping', f'L{i} Param: type_map', c, '--', 'X'),
            (f'L{i}::param::coarse_grain_layer', f'L{i} Param: coarse_grain', c, '-.', '*'),
        ]
        for key, label, color, ls, marker in grad_traces:
            y = _series_from_grad_history(key)
            if np.all(np.isnan(y)):
                continue
            any_plotted = True
            plt.plot(
                x, y, label=label, color=color, linestyle=ls, marker=marker,
                linewidth=1.8, markersize=4, markevery=max(1, len(x) // 20)
            )

    if any_plotted:
        plt.yscale('log')
        plt.xlabel('Epoch')
        plt.ylabel('Gradient norm (log scale)')
        plt.title('Gradient History (Hierarchical Layers)')
        plt.grid(True, which='both', linestyle='--', alpha=0.35)
        plt.legend(ncol=2, fontsize=9)
        grad_curve_path = os.path.join(SAVE_DIR, 'grad_history_curves.png')
        plt.savefig(grad_curve_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Gradient curves saved to {grad_curve_path}")
    else:
        print('No finite gradient history values to plot yet.')
    plt.close(fig)

# ── Final coarse-grained viz ──
if '_final_per_layer' in dir() and _final_per_layer:
    # Build per-layer data arrays for final viz
    final_graph_data = [(viz_z_0, viz_pos_0, viz_adj_0, viz_edge_dists_0)]
    final_spatial_labels = []
    final_cluster_com = []
    final_coarse_adj = []

    # Layer 0
    s_idx_0 = torch.argmax(_final_per_layer[0]['s_hard'], dim=-1)
    final_spatial_labels.append(s_idx_0[0][viz_mask[0].cpu()].numpy())
    final_cluster_com.append(_final_per_layer[0]['cluster_com'].numpy())
    final_coarse_adj.append(_final_per_layer[0]['coarse_adj'].numpy())

    # Subsequent layers
    for i in range(1, n_layers):
        m_prev = _final_per_layer[i - 1]['cg_mask']
        n_cg = int(m_prev.sum())
        cg_z   = _final_per_layer[i - 1]['cg_z'][:n_cg].numpy()
        cg_pos = _final_per_layer[i - 1]['cg_pos'][:n_cg].numpy()
        cg_adj = _final_per_layer[i - 1]['cg_adj'][:n_cg, :n_cg].numpy()
        final_graph_data.append((cg_z, cg_pos, cg_adj, None))

        m_cur = _final_per_layer[i]['cg_mask']
        s_idx_i = torch.argmax(_final_per_layer[i]['s_hard'], dim=-1)
        final_spatial_labels.append(s_idx_i[0][m_prev].numpy())
        final_cluster_com.append(_final_per_layer[i]['cluster_com'].numpy())
        final_coarse_adj.append(_final_per_layer[i]['coarse_adj'].numpy())

    save_coarse_grained_viz_multilayer(final_graph_data, final_spatial_labels,
                                       final_cluster_com, final_coarse_adj, SAVE_DIR)
    del _final_per_layer

# ── GIFs ──
all_gif_jobs = [
    (image_paths,          SAVE_DIR, 'training_evolution'),
    (mol_vs_c_h_paths,     None,     'mol_vs_c_h_evolution'),
    (node_feat_heatmap_paths, None,  'node_feat_evolution'),
]
for key, dir_name in [('s_space_heatmap', 's_space_evolution'),
                       ('adj_heatmap', 'adj_evolution'),
                       ('c_h_heatmap', 'c_h_evolution'),
                       ('s_type_heatmap', 's_type_evolution'),
                       ('c_soft_heatmap', 'c_soft_evolution'),
                       ('c_prime_heatmap', 'c_prime_evolution')]:
    all_gif_jobs.append((heatmap_path_histories.get(key, []), None, dir_name))

for paths, explicit_dir, gif_name in all_gif_jobs:
    if not paths:
        continue
    d = explicit_dir if explicit_dir else os.path.dirname(paths[0])
    gp = os.path.join(d, f'{gif_name}.gif')
    save_gif(paths, gp, fps=1, max_width=800)
    gc.collect()

print("All post-training outputs generated successfully.")

Model parameters reset.
GPU memory before training: 123.8 MB allocated, 176.0 MB reserved
Cleaning previously saved visualizations in mincut_viz...
Starting training... (press Ctrl+C once to stop gracefully, twice to force)
Epoch 1/250 | Total Loss: 1.3561
  L0 | Cut:-0.2428 Ortho:1.2410 Ent:0.0757 | Rec:0.0000 CutP:0.0000 OrthoP:0.0000 LinkP:0.0000 EntP:0.0000
  L1 | Cut:-0.0474 Ortho:0.3294 Ent:0.0003 | Rec:0.0000 CutP:0.0000 OrthoP:0.0000 LinkP:0.0000 EntP:0.0000
  Grad | Activation norms (first batch):
    L0: h=0.0002853151236195117 | s_type=0.04427681490778923 | softbfs=9.931556996889412e-05 | s_space=9.931556996889412e-05
    L1: h=1.0390653187641874e-05 | s_type=0.007176422514021397 | softbfs=None | s_space=None
  Grad | Parameter mean norms:
    L0: enc.embed=0.0028332958463579416 | enc.inter=0.005466828670318204 | type=0.014838769420748577 | softBFS=None | coarse=None
    L1: enc.embed=None | enc.inter=1.298076016677509e-05 | type=0.00020966647116438253 | softBFS=None | coars